In [1]:
import fitz 
import re
import json
import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CH4_PATH = RAW_DIR / "iesc104.pdf"
CH6_PATH = RAW_DIR / "iesc106.pdf"

print("Setup complete ")
print(f"Ch4 exists: {CH4_PATH.exists()}")
print(f"Ch6 exists: {CH6_PATH.exists()}")

Setup complete 
Ch4 exists: True
Ch6 exists: True


In [ ]:
#PDF Extraction form the chapters in the raw
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()
        pages.append({
            "page": page_num,
            "text": text.strip()
        })
    doc.close()
    return pages

ch4_pages = extract_text_from_pdf(CH4_PATH)
ch6_pages = extract_text_from_pdf(CH6_PATH)

print(f"Ch4 — Pages extracted: {len(ch4_pages)}")
print(f"Ch6 — Pages extracted: {len(ch6_pages)}")

# Quick sanity check — print first 300 chars of page 2
print("\n── Ch4 Page 2 preview ──")
print(ch4_pages[1]["text"][:300])

print("\n── Ch6 Page 2 preview ──")
print(ch6_pages[1]["text"][:300])

Ch4 — Pages extracted: 24
Ch6 — Pages extracted: 22

── Ch4 Page 2 preview ──
Describing Motion Around Us
49
about some more physical quantities, such as displacement, average 
velocity and average acceleration. You will also learn to describe motion 
not only in words, but also with numbers, equations and graphs.
4.1 Motion in a Straight Line
You have learnt that when an obj

── Ch6 Page 2 preview ──
How Forces Affect Motion
95
Fig. 6.3: (a) Pushing a box 
kept on table or floor
Force by hand
Force of friction
Fig. 6.2: Measuring the 
weight of an object using a 
spring balance
While learning about force earlier, did you notice that whenever any 
type of force acting on an object was described, 


In [ ]:
#Content Type Classification 
def classify_content_type(text):
    text_lower = text.lower()
    
    if re.search(r'example\s+\d+[\.:]\s*|answer\s*:', text_lower):
        return "worked_example"
    
    if re.search(r'revise,\s*reflect|pause and ponder|journey beyond|at a glance', text_lower):
        return "end_of_chapter"
    
    if re.search(r'activity\s+\d+[\.:]\s*', text_lower):
        return "activity"
    
    return "concept"

def split_into_sections(pages, chapter_name):
    sections = []
    current_section = "Introduction"
    
    for page_data in pages:
        text = page_data["text"]
        page_num = page_data["page"]
        
        # Detect section headings like 4.1, 4.2, 6.1 etc.
        heading_match = re.search(r'\n(\d+\.\d+[\.\d]*\s+[A-Z][^\n]{5,60})', text)
        if heading_match:
            current_section = heading_match.group(1).strip()
        
        content_type = classify_content_type(text)
        
        sections.append({
            "chapter": chapter_name,
            "page": page_num,
            "section": current_section,
            "content_type": content_type,
            "text": text
        })
    
    return sections

ch4_sections = split_into_sections(ch4_pages, "Chapter 4 - Describing Motion Around Us")
ch6_sections = split_into_sections(ch6_pages, "Chapter 6 - How Forces Affect Motion")

all_sections = ch4_sections + ch6_sections

df = pd.DataFrame(all_sections)
print("Content type distribution:")
print(df["content_type"].value_counts())
print(f"\nTotal pages classified: {len(all_sections)}")
print("\nSample section labels (Ch4):")
for s in ch4_sections[:5]:
    print(f"  Page {s['page']} | {s['content_type']:15} | {s['section'][:50]}")

Content type distribution:
content_type
concept           14
activity          13
worked_example    12
end_of_chapter     7
Name: count, dtype: int64

Total pages classified: 46

Sample section labels (Ch4):
  Page 1 | concept         | Introduction
  Page 2 | concept         | 4.1 Motion in a Straight Line
  Page 3 | concept         | 4.1.2 Distance travelled and displacement
  Page 4 | end_of_chapter  | 4.1.2 Distance travelled and displacement
  Page 5 | worked_example  | 4.1.3 Average speed and average velocity


In [ ]:
#Tokenizer Comparison
from transformers import AutoTokenizer

# i have used these three tokenizer 
gpt2_tokenizer    = AutoTokenizer.from_pretrained("gpt2")
bert_tokenizer    = AutoTokenizer.from_pretrained("bert-base-uncased")
t5_tokenizer      = AutoTokenizer.from_pretrained("t5-small")

# 5 representative passages from our corpus
passages = [
    "Displacement is the net change in the position of an object between the two given instants of time.",
    
    "The average velocity of an object in a time interval is given by v_av = s/t, where s is displacement and t is time interval. The SI unit is metre per second (m/s).",
    
    "The kinematic equations for motion in a straight line with constant acceleration are: v = u + at, s = ut + (1/2)at², and v² = u² + 2as.",
    
    "Newton's second law of motion states: when a net force acts on an object, the object accelerates in the direction of the net force. Mathematically, F = ma.",
    
    "When brakes are applied to a moving vehicle, it moves some distance before coming to a stop. The distance depends upon the velocity, road surface, and braking capacity of the vehicle."
]

results = []
print(f"{'Passage':<6} {'GPT2-BPE':>10} {'BERT-WP':>10} {'T5-SP':>10}  Disagreement Example")
print("-" * 80)

for i, passage in enumerate(passages, 1):
    gpt2_tokens = gpt2_tokenizer.tokenize(passage)
    bert_tokens = bert_tokenizer.tokenize(passage)
    t5_tokens   = t5_tokenizer.tokenize(passage)
    
    g, b, t = len(gpt2_tokens), len(bert_tokens), len(t5_tokens)
    
    sci_words = ["displacement", "acceleration", "kinematic", "velocity", "v²"]
    disagreement = ""
    for word in sci_words:
        if word.lower() in passage.lower():
            gw = [tok for tok in gpt2_tokens if word[:4].lower() in tok.lower()]
            bw = [tok for tok in bert_tokens if word[:4].lower() in tok.lower()]
            if gw or bw:
                disagreement = f"'{word}' → GPT2:{gw[:2]} BERT:{bw[:2]}"
                break
    
    print(f"P{i:<5} {g:>10} {b:>10} {t:>10}  {disagreement[:55]}")
    results.append({"passage": i, "gpt2": g, "bert": b, "t5": t})

print("\n── Detailed token boundaries for Passage 3 (formula-heavy) ──")
p3 = passages[2]
print(f"\nGPT2 tokens: {gpt2_tokenizer.tokenize(p3)}")
print(f"\nBERT tokens: {bert_tokenizer.tokenize(p3)}")
print(f"\nT5   tokens: {t5_tokenizer.tokenize(p3)}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

c:\Users\voidX\OneDrive\Desktop\parishiksha_study_assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\voidX\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

c:\Users\voidX\OneDrive\Desktop\parishiksha_study_assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\voidX\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passage   GPT2-BPE    BERT-WP      T5-SP  Disagreement Example
--------------------------------------------------------------------------------
P1             22         20         22  'displacement' → GPT2:[] BERT:['displacement']
P2             43         44         50  'displacement' → GPT2:['Ġdisplacement'] BERT:['displace
P3             44         44         52  'acceleration' → GPT2:['Ġacceleration'] BERT:['accelera
P4             37         37         40  
P5             36         36         40  'velocity' → GPT2:['Ġvelocity'] BERT:['velocity']

── Detailed token boundaries for Passage 3 (formula-heavy) ──

GPT2 tokens: ['The', 'Ġk', 'inem', 'atic', 'Ġequations', 'Ġfor', 'Ġmotion', 'Ġin', 'Ġa', 'Ġstraight', 'Ġline', 'Ġwith', 'Ġconstant', 'Ġacceleration', 'Ġare', ':', 'Ġv', 'Ġ=', 'Ġu', 'Ġ+', 'Ġat', ',', 'Ġs', 'Ġ=', 'Ġut', 'Ġ+', 'Ġ(', '1', '/', '2', ')', 'at', 'Â²', ',', 'Ġand', 'Ġv', 'Â²', 'Ġ=', 'Ġu', 'Â²', 'Ġ+', 'Ġ2', 'as', '.']

BERT tokens: ['the', 'kin', '##ema', '##tic', 'e

In [ ]:
# Chunking Strategy, started with chunk_size=300
def chunk_sections(sections, chunk_size=300, overlap=50):
    """
    Chunking strategy:
    - concept pages: 300 tokens, 50 overlap
    - worked_example: kept whole (problem + solution must stay together)
    - end_of_chapter: split into individual questions
    - activity: kept whole
    
    Tokenizer: BERT WordPiece (used consistently for both index and query)
    """
    chunks = []
    chunk_id = 0
    words_per_token = 0.75  

    for section in sections:
        text = section["text"]
        ctype = section["content_type"]
        
        # Clean text
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'Chapter-\d+\.indd.*', '', text).strip()
        
        if ctype == "worked_example":
            chunks.append({
                "chunk_id": chunk_id,
                "chapter": section["chapter"],
                "section": section["section"],
                "content_type": ctype,
                "page": section["page"],
                "text": text
            })
            chunk_id += 1

        elif ctype == "end_of_chapter":
            questions = re.split(r'(?=\n?\d+\.\s+[A-Z])', text)
            for q in questions:
                q = q.strip()
                if len(q) > 50:
                    chunks.append({
                        "chunk_id": chunk_id,
                        "chapter": section["chapter"],
                        "section": section["section"],
                        "content_type": ctype,
                        "page": section["page"],
                        "text": q
                    })
                    chunk_id += 1

        else:
            words = text.split()
            step = int(chunk_size * words_per_token)
            window = int((chunk_size + overlap) * words_per_token)
            
            if len(words) <= window:
                chunks.append({
                    "chunk_id": chunk_id,
                    "chapter": section["chapter"],
                    "section": section["section"],
                    "content_type": ctype,
                    "page": section["page"],
                    "text": text
                })
                chunk_id += 1
            else:
                for start in range(0, len(words), step):
                    chunk_text = " ".join(words[start:start + window])
                    if len(chunk_text) > 100:
                        chunks.append({
                            "chunk_id": chunk_id,
                            "chapter": section["chapter"],
                            "section": section["section"],
                            "content_type": ctype,
                            "page": section["page"],
                            "text": chunk_text
                        })
                        chunk_id += 1

    return chunks

chunks = chunk_sections(all_sections, chunk_size=300, overlap=50)

# Save to disk, find it in the processed folder 
with open(PROCESSED_DIR / "chunks.json", "w") as f:
    json.dump(chunks, f, indent=2)

# Summary
chunks_df = pd.DataFrame(chunks)
print(f"Total chunks created: {len(chunks)}")
print(f"\nChunks by content type:")
print(chunks_df["content_type"].value_counts())
print(f"\nChunks by chapter:")
print(chunks_df["chapter"].value_counts())
print(f"\nAvg chunk length (words): {chunks_df['text'].apply(lambda x: len(x.split())).mean():.0f}")
print(f"\nSample chunk (worked_example):")
ex = chunks_df[chunks_df.content_type == "worked_example"].iloc[0]
print(f"  Section: {ex['section']}")
print(f"  Preview: {ex['text'][:200]}")

Total chunks created: 101

Chunks by content type:
content_type
activity          36
concept           32
end_of_chapter    21
worked_example    12
Name: count, dtype: int64

Chunks by chapter:
chapter
Chapter 6 - How Forces Affect Motion       51
Chapter 4 - Describing Motion Around Us    50
Name: count, dtype: int64

Avg chunk length (words): 225

Sample chunk (worked_example):
  Section: 4.1.3 Average speed and average velocity
  Preview: Exploration|Grade 9 52 Motion, i.e., a change in the position of an object, can be described in terms of the total distance travelled by the object and its displacement. But how can you describe how f


In [ ]:
#Chunk Cleaning
def clean_chunk_text(text):
    # Remove page numbers like "102 Exploration|Grade 9"
    text = re.sub(r'\d*\s*Exploration\|Grade\s*9\s*\d*', '', text)
    
    # Remove figure references like "Fig. 4.3:", "Fig. 6.11:"
    text = re.sub(r'Fig\.?\s*\d+\.\d+[a-z]?\s*:?[^\n]*', '', text)
    
    # Remove chapter file markers like "Chapter-04.indd 55 03-Apr-26"
    text = re.sub(r'Chapter-\d+\.indd.*', '', text)
    
    # Remove standalone page numbers (a number alone on a line)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)
    
    # Remove table references
    text = re.sub(r'Table\s+\d+\.\d+\s*:?[^\n]*', '', text)
    
    # Remove "Grade X Curiosity Chapter Y" cross-reference boxes
    text = re.sub(r'Grade\s+\d+\s*Curiosity\s*Chapter\s*\d+', '', text)
    
    # Remove repeated whitespace and blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'\s{2,}', ' ', text)
    
    return text.strip()

for chunk in chunks:
    chunk["text"] = clean_chunk_text(chunk["text"])

chunks = [c for c in chunks if len(c["text"].split()) > 30]

print(f"Chunks after cleaning: {len(chunks)}")

with open(PROCESSED_DIR / "chunks.json", "w") as f:
    json.dump(chunks, f, indent=2)

corpus_tokens = [simple_tokenize(chunk["text"]) for chunk in chunks]
bm25 = BM25Okapi(corpus_tokens)
print("BM25 index rebuilt on clean chunks ✓")

test_queries = [
    "What is displacement and how is it different from distance?",
    "State Newton's second law of motion with formula",
    "What happens to velocity in uniform circular motion?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    results = retrieve(query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"\n  Rank {i} | Score: {r['bm25_score']} | {r['content_type']} | Page {r['page']}")
        print(f"  Section: {r['section']}")
        print(f"  Preview: {r['text'][:150]}...")

Chunks after cleaning: 71
BM25 index rebuilt on clean chunks ✓

Query: What is displacement and how is it different from distance?

  Rank 1 | Score: 10.2229 | end_of_chapter | Page 21
  Section: 4.4 Motion in a Plane
  Preview: 1. My father went to a shop from home which is located at a distance of 250 m on a straight road. On reaching there, he discovered that he forgot to c...

  Rank 2 | Score: 9.1046 | activity | Page 9
  Section: 6.5 Newton’s Second Law of Motion
  Preview: .5 Newton’s Second Law of Motion You know that a force can set an object in motion, bring it to rest, or change its velocity. A change in velocity mea...

  Rank 3 | Score: 8.2554 | concept | Page 23
  Section: 4.4 Motion in a Plane
  Preview: by drawing a velocity-time graph for its motion. 15. Two cars A and B start moving with a constant acceleration from rest in a straight line. Car A at...

Query: State Newton's second law of motion with formula

  Rank 1 | Score: 11.2853 | end_of_chapter | Page 19
  Sect

In [ ]:
#Improved Retriever
def retrieve(query, top_k=3, boost_concept=True):
    query_tokens = simple_tokenize(query)
    scores = bm25.get_scores(query_tokens)
    
    if boost_concept:
        for i, chunk in enumerate(chunks):
            if chunk["content_type"] == "concept":
                scores[i] *= 1.3
            elif chunk["content_type"] == "worked_example":
                scores[i] *= 1.2
            elif chunk["content_type"] == "end_of_chapter":
                scores[i] *= 0.8  # slight penalty — these are questions not answers
    
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    
    results = []
    for idx in top_indices:
        chunk = chunks[idx].copy()
        chunk["bm25_score"] = round(scores[idx], 4)
        results.append(chunk)
    return results

test_queries = [
    "What is displacement and how is it different from distance?",
    "State Newton's second law of motion with formula",
    "What happens to velocity in uniform circular motion?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    results = retrieve(query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"\n  Rank {i} | Score: {r['bm25_score']} | {r['content_type']} | Page {r['page']}")
        print(f"  Section: {r['section']}")
        print(f"  Preview: {r['text'][:200]}...")


Query: What is displacement and how is it different from distance?

  Rank 1 | Score: 10.7321 | concept | Page 23
  Section: 4.4 Motion in a Plane
  Preview: by drawing a velocity-time graph for its motion. 15. Two cars A and B start moving with a constant acceleration from rest in a straight line. Car A attains a velocity of 5 m s–1 in 5 s. Car B attains ...

  Rank 2 | Score: 9.8869 | concept | Page 16
  Section: 4.3 Kinematic Equations for Motion in a Straight
  Preview: Describing Motion Around Us 63 This is the displacement of the car from the origin in 6 seconds. You have found that the area enclosed by the velocity-time graph and the time axis for a desired time i...

  Rank 3 | Score: 9.3237 | worked_example | Page 5
  Section: 4.1.3 Average speed and average velocity
  Preview: Motion, i.e., a change in the position of an object, can be described in terms of the total distance travelled by the object and its displacement. But how can you describe how fast or slow an object i.

In [13]:
%pip install google-generativeai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Gemini Generation  used gemini free api', used 2.5flash
import os
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-flash")

GROUNDING_PROMPT = """You are a science study assistant for Class 9 students using NCERT textbooks.

You will be given CONTEXT extracted from NCERT Class 9 Science chapters.
Your job is to answer the student's question STRICTLY using only the provided context.

STRICT RULES:
1. If the answer is present in the context, answer clearly and simply for a Class 9 student.
2. If the answer is NOT in the context, respond with exactly: "I'm sorry, this topic is not covered in the provided chapter content. Please refer to your teacher or other chapters."
3. Do NOT use any outside knowledge, even if you know the answer.
4. Do NOT make up formulas, definitions, or examples not present in the context.
5. Keep answers concise — 3 to 5 sentences maximum.
6. If a formula is in the context, include it in your answer.

CONTEXT:
{context}

STUDENT QUESTION:
{question}

ANSWER:"""

def format_context(retrieved_chunks):
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        parts.append(
            f"[Source {i} | {chunk['chapter']} | {chunk['section']} | Page {chunk['page']}]\n"
            f"{chunk['text']}"
        )
    return "\n\n".join(parts)

def answer(question, top_k=3, temperature=0.0):
    retrieved = retrieve(question, top_k=top_k)
    
    context = format_context(retrieved)
    
    prompt = GROUNDING_PROMPT.format(context=context, question=question)
    
    response = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            temperature=0.0,
            max_output_tokens=1024
        )
    )

    if response.candidates[0].finish_reason.name != "STOP":
        print(f"WARNING: Generation stopped due to: {response.candidates[0].finish_reason.name}")
    
    return {
        "question": question,
        "answer": response.text.strip(),
        "retrieved_chunks": retrieved,
        "context_used": context
    }


#Quick smoke test
test_q = "What is Newton's second law of motion?"
result = answer(test_q)

print(f"Question: {result['question']}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nRetrieved from:")
for c in result['retrieved_chunks']:
    print(f"  - {c['section']} | Page {c['page']} | Score {c['bm25_score']}")

Question: What is Newton's second law of motion?

Answer:
Newton's second law of motion states that when a net force acts on an object, the object accelerates in the direction of the net force. The magnitude of this acceleration is proportional to the magnitude of the net force and is inversely proportional to the mass of the object. A more complete form of Newton's second law states that the rate of change of momentum of an object is proportional to the net force and occurs in the direction of the net force. Momentum is defined as the product of an object's mass and velocity.

Retrieved from:
  - 6.7 Forces Acting on a System of Objects | Page 19 | Score 12.7007
  - 6.5 Newton’s Second Law of Motion | Page 11 | Score 12.2819
  - 6.6 Newton’s Third Law of Motion | Page 15 | Score 9.777
